In [ ]:
# Imports and XML loading
import re
import xml.etree.ElementTree as ET

# Parse the XML file
tree = ET.parse('your_file.xml')
root = tree.getroot()


for child in root:
    print(child.tag, child.attrib)

In [ ]:
# Defining helper functions and extracting annotations

def format_time(milliseconds):
    """Convert milliseconds to HH:MM:SS.mmm format."""
    seconds = milliseconds // 1000
    ms = milliseconds % 1000
    minutes = seconds // 60
    hours = minutes // 60
    return f"{hours:02}:{minutes % 60:02}:{seconds % 60:02}.{ms:03}"


# Create a mapping of TIME_SLOT_ID to TIME_VALUE
time_slots = {
    ts.attrib['TIME_SLOT_ID']: int(ts.attrib['TIME_VALUE'])
    for ts in root.findall('.//TIME_SLOT')
}

# Extract all ALIGNABLE_ANNOTATION elements with their IDs, values, start, end times, and TIER_ID
annotations = []
for tier in root.findall('.//TIER'):
    tier_id = tier.attrib.get('TIER_ID')
    for annotation in tier.findall('.//ALIGNABLE_ANNOTATION'):
        annotation_id = annotation.attrib.get('ANNOTATION_ID')
        annotation_value = annotation.find('ANNOTATION_VALUE')
        time_ref1 = annotation.attrib.get('TIME_SLOT_REF1')
        time_ref2 = annotation.attrib.get('TIME_SLOT_REF2')

        annotation_value_text = annotation_value.text if annotation_value is not None else "No Value"
        start_time = time_slots.get(time_ref1, "No Start Time")
        end_time = time_slots.get(time_ref2, "No End Time")

        annotations.append((annotation_id, annotation_value_text, start_time, end_time, tier_id))

In [ ]:
# Sort annotations and print them

sorted_annotations = sorted(annotations, key=lambda x: x[2])

# Identify missing annotation IDs
missing_annotations = []
last_id = 0
for annotation in sorted_annotations:
    current_id = int(annotation[0][1:])
    if current_id != last_id + 1:
        missing_annotations.append((last_id + 1, current_id - 1))
    last_id = current_id

print("Sorted Annotations:")
for annotation_id, annotation_value_text, start_time, end_time, tier_id in sorted_annotations:
    print(
        f"ID: {annotation_id}, Value: {annotation_value_text}, "
        f"Start: {format_time(start_time)}, End: {format_time(end_time)}, Who: {tier_id}"
    )

In [ ]:
# Show missing annotation IDs and their time frames

print("\nMissing Annotations and Time Frames:")
for start_missing, end_missing in missing_annotations:
    try:
        start_time = (
            sorted_annotations[start_missing - 2][3]
            if start_missing - 2 >= 0 else 0
        )
        end_time = (
            sorted_annotations[end_missing][2]
            if end_missing < len(sorted_annotations) else sorted_annotations[-1][3]
        )

        print(
            f"Missing IDs: a{start_missing} to a{end_missing}, "
            f"Time Frame: {format_time(start_time)} - {format_time(end_time)}"
        )
    except IndexError:
        print(f"Skipping missing range a{start_missing} to a{end_missing} due to index issues.")

In [ ]:
# Group annotations by start and end time

grouped_annotations = {}
for annotation in annotations:
    key = (annotation[2], annotation[3])
    grouped_annotations.setdefault(key, []).append(annotation)

print("Grouped Annotations:")
for (start_time, end_time), group in grouped_annotations.items():
    print(f"Start: {format_time(start_time)}, End: {format_time(end_time)}, Who: {tier_id}")
    for annotation_id, annotation_value_text, _, _, tier_id in group:
        print(f"  ID: {annotation_id}, Value: {annotation_value_text}")
    print('─' * 150)

In [ ]:
# Subtitle helper functions

shift_ms = 0  # replace with your desired time shift in milliseconds

def format_srt_time(milliseconds):
    hours = milliseconds // 3600000
    minutes = (milliseconds % 3600000) // 60000
    seconds = (milliseconds % 60000) // 1000
    millis = milliseconds % 1000
    return f"{hours:02}:{minutes:02}:{seconds:02},{millis:03}"

def merge_annotations_no_overlap(annotations, shift_ms):
    annotations.sort(key=lambda x: x[2])
    combined_entries = []
    language1_entries = []
    language2_entries = []
    current_end_time = -1
    i = 0

    while i < len(annotations):
        original_start, original_end = annotations[i][2], annotations[i][3]
        start_time, end_time = original_start + shift_ms, original_end + shift_ms
        group = [annotations[i]]
        i += 1

        while (i < len(annotations) and
               annotations[i][2] == original_start and annotations[i][3] == original_end):
            group.append(annotations[i])
            i += 1

        if start_time <= current_end_time:
            start_time = current_end_time + 1
            end_time = max(end_time, start_time + 500)

        texts_combined = []
        language1_entries = []
        language2_entries = []

        for ann in group:
            tier_id_clean = ann[4].replace('ft@ ', '').replace('tx@ ', '')
            annotation_text = ann[1]

            if ann[4].startswith('ft@'):
                annotation_text = f"<i>{annotation_text}</i>"
                texts_combined = f"{annotation_text}"
                language2_text = annotation_text
            else:
                texts_combined = f"{tier_id_clean}: {annotation_text}"
                language1_text = f"{tier_id_clean}: {annotation_text}"

            if ann[4].startswith('tx@'):
                language1_text = f"{tier_id_clean}: {annotation_text}"
            if ann[4].startswith('ft@'):
                language2_text = annotation_text

        combined_entry = f"{len(srt_entries_combined) + 1}\n{format_srt_time(start_time)} --> {format_srt_time(end_time)}\n{language1_text}\n{language2_text}\n"
        combined_subtitles.append(combined_entry)

        language1_entry = f"{len(srt_entries_language1) + 1}\n{format_srt_time(start_time)} --> {format_srt_time(end_time)}\n{language1_text}\n"
        srt_entries_language1.append(language1_text)

        language2_entry = f"{len(srt_entries_language2) + 1}\n{format_srt_time(start_time)} --> {format_srt_time(end_time)}\n{language2_text}\n"
        srt_entries_language2.append(language2_entry)

        current_end_time = end_time

    return combined_entry, srt_entries_language1, srt_entries_language2

def update_srt_content(srt_content):
    name_abbreviations = {
        r"Speaker1": "S1",
        r"Speaker2": "S2",
        r"Speaker3": "S3"
    }
    for name, abbreviation in name_abbreviations.items():
        srt_content = re.sub(name, abbreviation, srt_content)
    return srt_content

In [ ]:
# Generate subtitle entries

sorted_annotations = sorted(annotations, key=lambda x: x[2])

srt_entries_combined = []
srt_entries_language1 = []
srt_entries_language2 = []

annotations.sort(key=lambda x: x[2])
current_end_time = -1
i = 0

while i < len(annotations):
    original_start, original_end = annotations[i][2], annotations[i][3]
    start_time, end_time = original_start + shift_ms, original_end + shift_ms
    group = [annotations[i]]
    i += 1

    while (i < len(annotations) and
           annotations[i][2] == original_start and annotations[i][3] == original_end):
        group.append(annotations[i])
        i += 1

    if start_time <= current_end_time:
        start_time = current_end_time + 1
        end_time = max(end_time, start_time + 500)

    texts_combined = []
    text_language1 = ""
    text_language2 = ""

    for ann in group:
        tier_id_clean = ann[4].replace('ft@ ', '').replace('tx@ ', '')
        annotation_text = ann[1]
        if ann[4].startswith('ft@'):
            language2_text = f"<i>{annotation_text}</i>"
            texts_combined.append(language2_text)
        else:
            language1_text = f"{tier_id_clean}: {annotation_text}"
            texts_combined.insert(0, language1_text)

    combined_entry = (
        f"{len(srt_entries_combined) + 1}\n"
        f"{format_srt_time(start_time)} --> {format_srt_time(end_time)}\n"
        + "\n".join(texts_combined) + "\n"
    )
    srt_entries_combined.append(combined_entry)

    language1_entry = (
        f"{len(srt_entries_language1) + 1}\n"
        f"{format_srt_time(start_time)} --> {format_srt_time(end_time)}\n"
        f"{language1_text}\n"
    )
    srt_entries_language1.append(language1_entry)

    language2_entry = (
        f"{len(srt_entries_language2) + 1}\n"
        f"{format_srt_time(start_time)} --> {format_srt_time(end_time)}\n"
        f"{language2_text}\n"
    )
    srt_entries_language2.append(language2_entry)

    current_end_time = end_time

In [ ]:
# Write subtitle files

combined_content = update_srt_content("\n".join(srt_entries_combined) + "\n")
with open("language1+language2.srt", "w", encoding="utf-8-sig") as file:
    file.write(combined_content)

language1_content = update_srt_content("\n".join(srt_entries_language1) + "\n")
with open("language1.srt", "w", encoding="utf-8-sig") as file:
    file.write(language1_content)

language2_content = "\n".join(srt_entries_language2) + "\n"
with open("language2.srt", "w", encoding="utf-8-sig") as file:
    file.write(language2_content)

print("Subtitle files 'language1+language2.srt', 'language1.srt', and 'language2.srt' have been successfully created.")